# ZFIN → ZAPP: the fish lookup tables

**Scope: the easiest case only** — a curator who knows their allele(s) and their
wild-type background. Four critical pieces, each with the file it comes from, the
function that builds it, and its composition:

1. the **allele lookup table** (what the search box searches),
2. the **gene lookup table** (what a typed gene name resolves against),
3. the **alteration_type enumeration**,
4. the **zygosity enumeration**.

The functions here mirror the production service
(`zapp_atlas.api.services.zfin_lookup`) rule for rule, and the last section asserts
the two agree — so this report stays honest without re-implementing the API.

Re-run: `just fetch-zfin` (fresh downloads) then `just qc-report` (renders HTML).

In [1]:
import re
from collections import Counter

import pandas as pd

from zapp_atlas.settings import DEFAULT_ZFIN_DATA_DIR as DATA

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 40)


def read_tsv(filename, columns):
    """A ZFIN download: tab-separated, no header row. We name the leading
    columns we actually use and ignore everything to the right of them."""
    return pd.read_csv(DATA / filename, sep="\t", header=None,
                       names=columns, usecols=range(len(columns)),
                       dtype=str, quoting=3)


# The leading columns of each file we use, in file order.
FEATURE_COLS = ["allele_id", "so_type", "symbol", "symbol_long", "type_label",
                "mutagen", "treated_with", "construct_id", "construct_name"]
AFFECTED_GENE_COLS = ["allele_id", "so_type", "symbol",
                      "gene_symbol", "gene_id", "gene_so", "relationship"]
WILDTYPE_COLS = ["fish_id", "name", "abbreviation", "genotype_id"]
MARKER_COLS = ["gene_id", "symbol", "name", "marker_type", "so_type"]
GENOTYPE_COLS = ["genotype_id", "display_name", "identity"]

## Enumeration 1 — `alteration_type`

The values live in **one place: the schema YAML** (`SequenceAlterationTypeEnum`),
where each value carries its Sequence Ontology term as a `meaning:`. We invert that
into an SO → enum map, and every allele's SO type either maps to a value or stays
`None` (the card then shows ZFIN's own label — not an error, the id still points at
the full record).

In [2]:
from linkml_runtime import SchemaView

from zapp_atlas.schema.constraints import SCHEMA_PATH

enum = SchemaView(str(SCHEMA_PATH)).get_enum("SequenceAlterationTypeEnum")
SO_TO_ALTERATION = {pv.meaning: value for value, pv in enum.permissible_values.items()
                    if pv.meaning}

pd.DataFrame(sorted(SO_TO_ALTERATION.items()), columns=["SO term", "alteration_type value"])

/Users/ao33/Desktop/ZAPP_DATA_MODEL_FISH_UPDATE/zapp-atlas/server/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,SO term,alteration_type value
0,SO:0000159,deletion
1,SO:0000667,insertion
2,SO:0001059,sequence_alteration
3,SO:0001218,transgenic_insertion
4,SO:1000002,substitution
5,SO:1000005,complex_substitution
6,SO:1000008,point_mutation
7,SO:1000032,indel
8,SO:1000035,duplication
9,SO:1000036,inversion


## Enumeration 2 — `zygosity`

Also defined in the schema YAML (`ZygosityEnum`). Zygosity is always the **curator's
assertion** — nothing looks it up. ZFIN speaks the same vocabulary in its genotype
identity strings, where every allele carries a `[fish, mother, father]` code triple:

In [3]:
from zapp_atlas.schema.pydantic_crud import ZygosityEnum

ZFIN_CODE_TO_OURS = {"2": "homozygous", "1": "heterozygous",
                     "U": "unknown", "W": "wild_type"}

print("our values:", [value.value for value in ZygosityEnum])


def observed_zygosity_codes():
    """Count the [fish,mother,father] codes across all registered genotypes."""
    counts = {"fish": Counter(), "mother": Counter(), "father": Counter()}
    for identity in read_tsv("genotype_features.txt", GENOTYPE_COLS)["identity"].dropna():
        for triple in re.findall(r"\[([^\]]*)\]", identity):
            codes = [part.strip() for part in triple.split(",")]
            if len(codes) == 3:
                for who, code in zip(counts, codes):
                    counts[who][code] += 1
    table = pd.DataFrame(counts).fillna(0).astype(int)
    table.insert(0, "our value", table.index.to_series().map(ZFIN_CODE_TO_OURS)
                 .fillna("— not modeled —"))
    return table


observed_zygosity_codes()

our values: ['homozygous', 'heterozygous', 'unknown', 'wild_type']


,our value,fish,mother,father
U,unknown,44541,66241,66692
2,homozygous,36864,2557,1679
1,heterozygous,14256,26707,26985
C,— not modeled —,23,3,1
W,wild_type,0,176,327


Read of that table: parents are mostly **U (unknown)** — so the form defaults
parents to `unknown` — and **W (wild type)** is routine for parents, which is why
`wild_type` was added to the enum. A fish is never recorded W for an allele it
carries (a wild-type fish simply doesn't list the allele). The rare **C (complex)**
code (~27 occurrences) is deliberately not modeled yet; if that row grows, revisit.

## Lookup table 1 — wild-type backgrounds

**File:** `wildtypes_fish.txt`, one row per wild-type line. This *is* the background
dropdown: pick a name, and the strain's genotype id (and, for a pure wild-type fish,
its fish id) attach. The form adds one non-ZFIN option, **"unknown"**, stored as an
omitted field — no record is ever guessed. Stored ids get a `ZFIN:` prefix (CURIE).

In [4]:
def load_wildtypes():
    """The background dropdown, straight from the file."""
    return read_tsv("wildtypes_fish.txt", WILDTYPE_COLS).sort_values("name",
        key=lambda names: names.str.lower()).reset_index(drop=True)


wildtypes = load_wildtypes()
wildtypes

,fish_id,name,abbreviation,genotype_id
0,ZDB-FISH-150901-27842,AB,AB,ZDB-GENO-960809-7
1,ZDB-FISH-150901-19012,AB/C32,AB/C32,ZDB-GENO-070425-3
2,ZDB-FISH-150901-18519,AB/EKW,AB/EKW,ZDB-GENO-091223-1
3,ZDB-FISH-150901-29235,AB/TL,AB/TL,ZDB-GENO-031202-1
4,ZDB-FISH-150901-29084,AB/TU,AB/TU,ZDB-GENO-010924-10
5,ZDB-FISH-181106-1,ABO,ABO,ZDB-GENO-181106-1
6,ZDB-FISH-150901-28222,C32,C32,ZDB-GENO-030501-1
7,ZDB-FISH-180717-1,Cooch Behar,CB,ZDB-GENO-180717-1
8,ZDB-FISH-150901-28033,DAR,DAR,ZDB-GENO-960809-13
9,ZDB-FISH-150901-29731,EKW,EKW,ZDB-GENO-990520-2


## Lookup table 2 — alleles

**Files:** `features.txt` (one row per allele *record* — but see below) joined with
`features-affected-genes.txt` (which gene an allele damages).

Two file quirks the functions handle, both verified against the data:

- `features.txt` **repeats ~1.7k alleles** — identically, or once per construct
  (a co-injected line is ONE insertion carrying several constructs) → aggregate
  per allele id.
- In `features-affected-genes.txt`, only rows whose relationship is
  **`is allele of`** link an allele to the gene it damages (the rest is curation
  bookkeeping) → keep only those; one gene per allele (just 2 of 59k have more).

In [5]:
SO_TRANSGENIC_INSERTION = "SO:0001218"


def load_affected_genes():
    """allele_id -> the gene it damages, one row per allele."""
    genes = read_tsv("features-affected-genes.txt", AFFECTED_GENE_COLS)
    genes = genes[genes["relationship"] == "is allele of"]
    return genes.drop_duplicates("allele_id")[["allele_id", "gene_symbol", "gene_id"]]


def load_alleles():
    """One row per allele: the table the search box searches."""
    rows = read_tsv("features.txt", FEATURE_COLS)
    rows = rows[rows["allele_id"].str.startswith("ZDB-ALT-", na=False)]

    constructs = (rows[["allele_id", "construct_id", "construct_name"]].dropna()
                  .drop_duplicates(["allele_id", "construct_id"])
                  .groupby("allele_id")
                  .agg(construct_name=("construct_name", " + ".join),
                       n_constructs=("construct_id", "size")))

    alleles = rows.drop_duplicates("allele_id")
    alleles = alleles[["allele_id", "symbol", "so_type", "type_label", "mutagen"]]
    alleles = alleles.merge(constructs, on="allele_id", how="left")
    alleles = alleles.merge(load_affected_genes(), on="allele_id", how="left")

    alleles["mutagen"] = alleles["mutagen"].replace("not specified", pd.NA)
    alleles["n_constructs"] = alleles["n_constructs"].fillna(0).astype(int)
    alleles["alteration_type"] = alleles["so_type"].map(SO_TO_ALTERATION)
    alleles["is_transgenic"] = ((alleles["so_type"] == SO_TRANSGENIC_INSERTION)
                                | (alleles["n_constructs"] > 0))
    return alleles


alleles = load_alleles()
alleles[alleles["symbol"].isin(["fh111", "w200Tg", "gz13Tg"])]

,allele_id,symbol,so_type,type_label,mutagen,construct_name,n_constructs,gene_symbol,gene_id,alteration_type,is_transgenic
585,ZDB-ALT-160602-14,fh111,SO:1000008,Allele with one point mutation,NaN,NaN,0,snapc1b,ZDB-GENE-040426-716,point_mutation,False
53144,ZDB-ALT-090312-1,gz13Tg,SO:0001218,Transgenic Insertion,DNA,Tg2(krt4:GFP) + Tg2(mylpfa:RFP),2,NaN,NaN,transgenic_insertion,True
75013,ZDB-ALT-130130-3,w200Tg,SO:0001218,Transgenic Insertion,DNA,Tg(mpeg1:YFP),1,NaN,NaN,transgenic_insertion,True


### Composition of the allele table

The same numbers, in the data model's terms: which card a search hit becomes
(`TransgenicAllele` vs `MutantAllele`) and how complete each card arrives.

In [6]:
transgenic = alleles["is_transgenic"]
placeholder = alleles["symbol"].str.endswith(("_unspecified", "_unrecovered"))
has_gene = alleles["gene_symbol"].notna()

pd.DataFrame([
    ["alleles, total", len(alleles)],
    ["→ TransgenicAllele cards", transgenic.sum()],
    ["    with construct name + id", (transgenic & alleles["construct_name"].notna()).sum()],
    ["    co-injected (2 constructs)", (alleles["n_constructs"] > 1).sum()],
    ["→ MutantAllele cards", (~transgenic).sum()],
    ["    with the damaged gene attached", (~transgenic & has_gene).sum()],
    ["    gene unknown (shown honestly)", (~transgenic & ~has_gene).sum()],
    ["    gene-level placeholders (_unspecified / _unrecovered)", placeholder.sum()],
    ["alteration_type mapped to the enum", alleles["alteration_type"].notna().sum()],
    ["label-only (unmapped SO type)", alleles["alteration_type"].isna().sum()],
], columns=["what", "count"])

,what,count
0,"alleles, total",81170
1,→ TransgenicAllele cards,28583
2,with construct name + id,28293
3,co-injected (2 constructs),80
4,→ MutantAllele cards,52587
5,with the damaged gene attached,52145
6,gene unknown (shown honestly),442
7,gene-level placeholders (_unspecified / _unrecovered),176
8,alteration_type mapped to the enum,80617
9,label-only (unmapped SO type),553


## Lookup table 3 — genes

**File:** `genetic_markers.txt` — every marker ZFIN tracks, one row each, with a
`marker_type`.

**Namespace:** we use **ZFIN official gene symbols** for humans (ZFIN is the
zebrafish nomenclature authority — it *assigns* the symbols) and **ZDB-GENE ids**
(stored as `ZFIN:` CURIEs) as the identifier. Symbols are current as of the
download; historical symbols live in `aliases.txt`, so renames never orphan a
search.

**Which types count as a "gene":** real alleles damage more than protein-coding
genes — 157 of the gene ids referenced by `features-affected-genes.txt` are miRNA
genes, lincRNA genes, pseudogenes, or regulatory regions. So the supported universe
is the **gene-like set** below (100% referential integrity), not `GENE` alone
(99.23%).

In [7]:
GENE_LIKE = {
    "GENE": "protein-coding gene",
    "GENEP": "pseudogene",
    "MIRNAG": "microRNA gene",
    "LINCRNAG": "lincRNA gene",
    "LNCRNAG": "lncRNA gene",
    "NCRNAG": "ncRNA gene",
    "SNORNAG": "snoRNA gene",
    "ENHANCER": "enhancer",
    "NCCR": "non-coding control region",
}


def load_genes():
    """The searchable gene universe: every gene-like ZFIN marker."""
    markers = read_tsv("genetic_markers.txt", MARKER_COLS)
    genes = markers[markers["marker_type"].isin(GENE_LIKE)]
    return genes[["gene_id", "symbol", "name", "marker_type"]]


genes = load_genes()

breakdown = genes["marker_type"].value_counts().rename_axis("marker_type").reset_index(name="genes")
breakdown["meaning"] = breakdown["marker_type"].map(GENE_LIKE)
print(f"{len(genes):,} searchable genes")
display(breakdown)

referenced = load_affected_genes()["gene_id"]
print(f"referential integrity: {referenced.isin(genes['gene_id']).mean():.2%} "
      "of gene ids referenced by alleles exist in this table")

38,347 searchable genes


,marker_type,genes,meaning
0,GENE,36049,protein-coding gene
1,LINCRNAG,915,lincRNA gene
2,MIRNAG,426,microRNA gene
3,GENEP,418,pseudogene
4,NCCR,209,non-coding control region
5,ENHANCER,186,enhancer
6,LNCRNAG,60,lncRNA gene
7,NCRNAG,51,ncRNA gene
8,SNORNAG,33,snoRNA gene


referential integrity: 100.00% of gene ids referenced by alleles exist in this table


## Cross-check: these tables agree with the production API

The service (`zfin_lookup.get_index`) builds its index by the same rules. If a
refactor ever changes one side, this cell fails.

In [8]:
from zapp_atlas.api.services.zfin_lookup import get_index

index = get_index(DATA)
assert index.allele_count == len(alleles)
assert len(index.wildtypes) == len(wildtypes)

fh111 = index.search_alleles("fh111", 1)[1][0]
row = alleles.set_index("symbol").loc["fh111"]
assert fh111.allele_id == "ZFIN:" + row["allele_id"]        # stored form is a CURIE
assert fh111.affected_genes[0].gene_symbol == row["gene_symbol"]
assert fh111.alteration_type.value == row["alteration_type"]

print("notebook tables agree with the production index ✓")

notebook tables agree with the production index ✓


## Decisions on record (2026-09 snapshot)

- **Aggregate `features.txt` per allele id** — 1,755 repeated ids; 80 co-injected
  lines carry two constructs under one allele (e.g. `gz13Tg`).
- **Gene link = `is allele of` rows only, one gene per allele** — 2 of 59k alleles
  touch more than one gene; scalar slots are fine.
- **Gene universe = the gene-like marker types** (protein-coding + ncRNA genes,
  pseudogenes, regulatory regions) — 157 referenced ids are non-`GENE`; integrity
  100% vs 99.23% for `GENE` alone. Symbols are ZFIN's official assignments; stored
  ids are `ZFIN:ZDB-GENE-…` CURIEs.
- **`alteration_type` values live in the schema YAML as SO meanings** — 99.3% of
  alleles map; the rest show ZFIN's label and remain valid.
- **Zygosity is a curator assertion** — parents default `unknown` (U dominates
  ZFIN's own records); `wild_type` = ZFIN's W; the rare C (complex, ~27×) is
  deliberately unmodeled until it matters.
- **"Unknown" background = omitted field** — absence is unambiguous; no guessed
  records.

---
*Regenerate: `just fetch-zfin && just qc-report`.*